In [ ]:
from recbole.quick_start import load_data_and_model, run_recbole
import torch
import pandas as pd
import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import NeuMF
from recbole.trainer import Trainer
from recbole.utils import get_model, get_trainer, init_seed, init_logger
from collections import defaultdict
import os
from recbole.quick_start import load_data_and_model
import rbo as rbo_lib
import rbo

## Loading  base model

In [ ]:
import torch
import numpy as np
import pandas as pd
from collections import defaultdict

# ── 0. LOAD MODEL AND DATASET ───────────────────────────────────────────────
from recbole.quick_start import load_data_and_model
config, base_model, dataset, train_data, valid_data, test_data = load_data_and_model(
    config_dict={'data_path': 'dataset/ml-1m', 'dataset': 'ml-1m'}, 
    model_file='./saved/sasrec_ml-1m.pth'
)
base_model.eval()
device = config['device']
## LightGCN-Oct-19-2026_08-45-31.pth (beeradvocate dataset)
## SASRec-Sep-04-2026_14-07-54.pth (new SASRec model for beeradvocate dataset)
## SASRec-Oct-05-2026_19-09-24.pth (new SASRec model for lastfm dataset new filtering)

## Loading CBM model

In [ ]:
import yaml
from recbole.quick_start import load_data_and_model

#with open('YAML_files/CBM_config.yaml', 'r') as f:
 #   config_overrides = yaml.safe_load(f)

#config_overrides['train_neg_sample_args']={'distribution': 'none', 'sample_num': 'none', 'alpha': 'none', 'dynamic': False, 'candidate_num': 0}

config, CBM_model, dataset, train_data, valid_data, test_data = load_data_and_model(
        config_dict={'data_path': 'dataset/ml-1m', 'dataset': 'ml-1m', 'base_path':'./saved/sasrec_ml-1m.pth'}, 
        model_file="saved/SASRec_CBM-Aug-22-2026_12-43-59.pth")



## SASRec_CBM-Oct-05-2026_19-44-25.pth (lastfm SASRec_CBM for new backbone model)


## SASRec_CBM-Oct-04-2026_14-51-55.pth (beeradvocate by droping regularization term for SASRec)
## SASRec_CBM-Oct-04-2026_13-56-57.pth (beeradvocate by droping reconstruction term for SASRec)
## SASRec_CBM-Oct-04-2026_12-58-56.pth (beeradvocate by droping concpet term for SASRec)

## SASRec_CBM-Sep-26-2026_16-35-00.pth (ml-1m SASRec for channel size 32)
## SASRec_CBM-Sep-26-2026_15-46-24.pth(ml-1m SASRec for channel size 16)
## SASRec_CBM-Sep-23-2026_15-18-41.pth (ml-1m SASRec for channel size 8 )


### SASRec_CBM-Sep-04-2026_15-12-14.pth (New CBM model for beeradvocate for filtering dataset at least 40 interactions and using chatgpt)
## SASRec_CBM-Sep-04-2026_14-28-37.pth (beeradvocate dataset for previous cloud filtering cocnepts)
## SASRec_CBM-Aug-24-2026_09-58-42.pth (lastfm droping regularization term)
## SASRec_CBM-Aug-24-2026_09-08-07.pth (lastfm droping reconstruction)
## SASRec_CBM-Aug-24-2026_08-25-51.pth ( lastfm droping concept part)
## SASRec_CBM-Aug-24-2026_07-40-38.pth (lastfm droping channel size)
## SASRec_CBM-Aug-23-2026_15-13-19.pth (ml-1m Droping reconstruction both temrs)
#SASRec_CBM-Aug-23-2026_13-36-26.pth"(ml-1m Droping regularization term for z)
## SASRec_CBM-Aug-22-2026_12-43-59.pth


## LightGCN_CBM-Aug-12-2026_12-19-07.pth (lastfm second)
## LightGCN_CBM-Aug-10-2026_13-53-03.pth (lastfm-not good)
## LightGCN_CBM-Aug-06-2026_16-49-14.pth (ml-1m)
## saved/LightGCN_CBM-Aug-06-2026_22-52-56.pth (lighgcn on ml-1m)

## SASRec_CBM-Jul-27-2026_21-09-27.pth (beeradvocate hidden size 64 and niche and pop 30 and 90 percent)
## SASRec_CBM-Jul-27-2026_10-26-47.pth(beeradvocate hidden size 64 and niche and pop only 10 percent)
## SASRec_CBM-Jul-20-2026_18-18-44.pth (i dont know)
## SASRec_CBM-Jul-20-2026_17-52-19.pth. (adding loss_suff term to the loss fucntion)
## SASRec_CBM-Jul-20-2026_12-58-13.pth (v2.1)
## SASRec_CBM-Jul-19-2026_21-22-31.pth (pop>80, niche <20 same structure as befor)
## SASRec_CBM-Jul-20-2026_10-38-21.pth (reg is 1 and for sigmoid function, free channel is 64)
## SASRec_CBM-Jul-19-2026_18-41-45.pth (for sigmoid function and free channel 64 is a good result regularization term (0.1) for n_free)
##SASRec_CBM-Jul-19-2026_18-04-46.pth" (for sigmoid function and free channel 64 is a good result)
## SASRec_CBM-Jul-16-2026_17-56-02.pth. (this one is with 64 free size and adding all terms in losss functiona nd with softmax actiation fucntion)
# SASRec_CBM-Jul-11-2026_15-59-58.pth
CBM_model.eval()
device = config['device']

In [ ]:
@torch.no_grad()
def get_top_k_recommendations(model, data_loader, k=20, exclude_seen=True):
    """
    Returns {internal_user_id: LongTensor[k]} of top-k recommended INTERNAL
    item ids, ranked best-first.

    Compatible with compare_recommendations(): each value is a tensor on which
    .numpy().tolist() works.

    exclude_seen=True masks items already in the user's history before ranking,
    matching RecBole's default full-sort eval protocol. Set False only if your
    reported Recall/NDCG were computed WITHOUT seen-item masking.
    """
    model.eval()
    device = next(model.parameters()).device
    recs = {}

    for batch in data_loader:
        interaction = batch[0] if isinstance(batch, (list, tuple)) else batch
        interaction = interaction.to(device)

        # [B, n_items] full-catalog scores
        scores = model.full_sort_predict(interaction)
        B = interaction[model.ITEM_SEQ].shape[0]
        scores = scores.view(B, -1).clone()

        # mask padding item (internal id 0)
        scores[:, 0] = -float('inf')

        if exclude_seen:
            item_seq = interaction[model.ITEM_SEQ]
            for i in range(B):
                seen = item_seq[i][item_seq[i] != 0]
                scores[i, seen] = -float('inf')

        topk = torch.topk(scores, k, dim=1).indices          # [B, k] internal ids

        uids = interaction[model.USER_ID].tolist()
        for u, row in zip(uids, topk):
            recs[u] = row.detach().cpu()

    return recs

## H and H_hat similarity

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import spearmanr


@torch.no_grad()

def collect_h_hhat(cbm_model, data_loader, device=None):
    """
    Run the CBM over a data loader and collect the frozen-SASRec embedding `h`
    and the reconstructed embedding `h_hat` for every batch.

    Returns two tensors: H [N, hidden], H_hat [N, hidden]  (on CPU).
    """
    if device is None:
        device = next(cbm_model.parameters()).device

    cbm_model.eval()
    H, H_hat = [], []

    for batch in data_loader:
        # RecBole loaders yield an Interaction (or a tuple where it's first)

        interaction = batch[0] if isinstance(batch, (list, tuple)) else batch
        interaction = interaction.to(device)

        item_seq     = interaction[cbm_model.ITEM_SEQ]
        item_seq_len = interaction[cbm_model.ITEM_SEQ_LEN]
        

        h, c_hat,z, h_hat = cbm_model.forward(item_seq, item_seq_len)

        #h, c_hat, h_hat = cbm_model.forward(item_seq, item_seq_len)
        H.append(h.detach().cpu())
        H_hat.append(h_hat.detach().cpu())

    return torch.cat(H, dim=0), torch.cat(H_hat, dim=0)


def embedding_similarity_report(H, H_hat, item_embeddings=None, n_score_sample=2000):
    """
    H, H_hat : [N, hidden] tensors (the same N rows, aligned).
    item_embeddings : optional [n_items, hidden] tensor — if given, also reports
                      how similarly h and h_hat *rank* items (Spearman of scores).

    Prints a report and returns a dict of metrics.
    """
    assert H.shape == H_hat.shape, f"shape mismatch {H.shape} vs {H_hat.shape}"
    N, D = H.shape
    Hf, Hhf = H.float(), H_hat.float()

    # ── 1. Per-row cosine similarity (direction) ─────────────────────────────
    cos = F.cosine_similarity(Hf, Hhf, dim=-1)              # [N]

  
    

    # ── 5. Coefficient of determination R^2 (variance explained) ─────────────
    ss_res = ((Hf - Hhf) ** 2).sum()
    ss_tot = ((Hf - Hf.mean()) ** 2).sum()

    metrics = {
        'cosine_mean':      cos.mean().item(),
        'cosine_median':    cos.median().item(),
        'cosine_std':       cos.std().item(),
      
    }

    print(f"\n{'='*60}")
    print(f"Embedding similarity:  h  vs  h_hat   (N={N}, dim={D})")
    print(f"{'='*60}")
    print(f"  Cosine    mean={metrics['cosine_mean']:.4f}  "
          f"median={metrics['cosine_median']:.4f}  "
          f"std={metrics['cosine_std']:.4f}")
   



    print(f"{'='*60}\n")
    return metrics

In [ ]:
H, H_hat = collect_h_hhat(CBM_model, test_data)

metrics = embedding_similarity_report(
    H, H_hat,
    item_embeddings=CBM_model.item_embedding.weight,   # ties it back to ranking
)

In [ ]:
def compare_recommendations(recs_sasrec, recs_cbm, k=20, p=0.9):
    """
    Compare top-k recommendations between two models.
    Computes per-user RBO, Jaccard, and overlap count.
    """
    rbo_scores     = []
    jaccard_scores = []
    overlap_counts = []
    
    common_users = set(recs_sasrec.keys()) & set(recs_cbm.keys())
    
    for user_idx in sorted(common_users):
        list_a = recs_sasrec[user_idx].numpy().tolist()
        list_b = recs_cbm[user_idx].numpy().tolist()
        
        set_a = set(list_a)
        set_b = set(list_b)
        
        intersection = len(set_a & set_b)
        union        = len(set_a | set_b)
        
        jaccard = intersection / union if union > 0 else 0
        rbo_val     = rbo.RankingSimilarity(list_a, list_b).rbo_ext(p=0.9)
        
        rbo_scores.append(rbo_val )
        jaccard_scores.append(jaccard)
        overlap_counts.append(intersection)
    
    rbo_scores     = np.array(rbo_scores)
    jaccard_scores = np.array(jaccard_scores)
    overlap_counts = np.array(overlap_counts)
    
    print(f"\n{'='*60}")
    print(f"Top-{k} Recommendation Comparison: SASRec vs CBM")
    print(f"{'='*60}")
    print(f"  Users compared:        {len(common_users)}")
    print(f"  Mean RBO (p={p}):       {rbo_scores.mean():.4f}")
    print(f"  Median RBO:            {np.median(rbo_scores):.4f}")
    print(f"  Mean Jaccard:          {jaccard_scores.mean():.4f}")
    print(f"  Median Jaccard:        {np.median(jaccard_scores):.4f}")
    print(f"  Mean overlap (/{k}):    {overlap_counts.mean():.1f}")
    print(f"  Median overlap (/{k}):  {np.median(overlap_counts):.1f}")
    print(f"  Perfect match (={k}):   {(overlap_counts == k).sum()} users")
    print(f"  Zero overlap (=0):     {(overlap_counts == 0).sum()} users")
    
    print(f"\n  Overlap distribution:")
    for threshold in [0, 5, 10, 15, 20]:
        count = (overlap_counts >= threshold).sum()
        pct = 100 * count / len(overlap_counts)
        print(f"    ≥{threshold:2d} items overlap:  {count:5d} users ({pct:.1f}%)")
    
    return rbo_scores, jaccard_scores, overlap_counts


recs_sasrec = get_top_k_recommendations(base_model, test_data, k=20)
recs_cbm    = get_top_k_recommendations(CBM_model, test_data, k=20)

rbo_scores, jaccard_scores, overlap_counts = compare_recommendations(recs_sasrec, recs_cbm, k=10, p=0.9)

# Sweeping

In [ ]:
def evaluate_with_recbole_sasrec(model, config, train_data, test_data):
    """Set steering on the model and run RecBole eval. Tier/genre/year exposure
    metrics are reported automatically alongside standard rec metrics."""
   

    

    trainer = Trainer(config, model)
    trainer.eval_collector = Collector(config)
    trainer.evaluator      = Evaluator(config)
    trainer.eval_collector.data_collect(train_data)

    result = trainer.evaluate(test_data, load_best_model=False, show_progress=False)

 
    return result
def evaluate_with_recbole(cbm, config, train_data, test_data,
                          concept_idx=None, scale=1.0):
    """Set steering on the model and run RecBole eval. Tier/genre/year exposure
    metrics are reported automatically alongside standard rec metrics."""
    cbm.steer_concept_idx = concept_idx
    #config['steer_concept_idx'] = concept_idx
    cbm.steer_scale       = scale
    #config['steer_scale']       = scale
   

    trainer = Trainer(config, cbm)
    trainer.eval_collector = Collector(config)
    trainer.evaluator      = Evaluator(config)
    trainer.eval_collector.data_collect(train_data)

    result = trainer.evaluate(test_data, load_best_model=False, show_progress=False)

    cbm.steer_concept_idx = None
    #config['steer_concept_idx'] = None
    cbm.steer_scale       = 1.0
    #config['steer_scale' ]      = 1.0
    return result

## Overlap and Exact

In [ ]:
import torch
import numpy as np

CBM_model.eval()
device = config['device']

# -------------------------------
# 1. Clear evaluation buffers
# -------------------------------
CBM_model._eval_c_hat_buffer = []
CBM_model._eval_gt_buffer = []

# -------------------------------
# 2. Run test set
# -------------------------------
with torch.no_grad():

    for batch in test_data:

        if isinstance(batch, tuple):
            interaction = batch[0]
        else:
            interaction = batch

    
        # Your full_sort_predict() automatically captures:
        # c_hat -> _eval_c_hat_buffer
        # gt    -> _eval_gt_buffer
        interaction = interaction.to(device)
        CBM_model.full_sort_predict(interaction)
        #CBM_model.forward()


# -------------------------------
# 3. Concatenate all users
# -------------------------------
c_hat = torch.cat(
    CBM_model._eval_c_hat_buffer,
    dim=0
).numpy()

gt = torch.cat(
    CBM_model._eval_gt_buffer,
    dim=0
).numpy()

print("Predicted concepts:", c_hat.shape)
print("Ground truth:", gt.shape)


# -------------------------------
# 4. Get ordered top-3 concepts
# -------------------------------
k = 3

pred_top = np.argsort(-c_hat, axis=1)[:, :k]
gt_top   = np.argsort(-gt, axis=1)[:, :k]


# -------------------------------
# 5. Ordered soft accuracy
# -------------------------------
soft_per_user = (pred_top == gt_top).mean(axis=1)
soft_accuracy = soft_per_user.mean()


# -------------------------------
# 6. Ordered strict accuracy
# -------------------------------
strict_per_user = np.all(
    pred_top == gt_top,
    axis=1
)

strict_accuracy = strict_per_user.mean()


# -------------------------------
# 7. Results
# -------------------------------
print(f"Concept Accuracy Soft:   {soft_accuracy:.4f}")
print(f"Concept Accuracy Strict: {strict_accuracy:.4f}")

In [ ]:
import types
import torch
import torch.nn.functional as F

from recbole.utils.build_concepts import build_lookups
from recbole.evaluator import Collector, Evaluator

dataset_name = "ml-1m"

# ═══════════════════════════════════════════════════════════════
# Patched forward: multi-concept intervention + activation capture
# ═══════════════════════════════════════════════════════════════
def patched_forward(self, item_seq, item_seq_len):
    h = self._encode(item_seq, item_seq_len)
    #c = self._lookup_concepts(item_seq)

    #c_hat = self.concept_predictor(h)
    logits = self.concept_predictor(h)
    c_hat=torch.sigmoid(logits)
    #print("c_hat_bef_pop:", c_hat_tmp[:,18].mean())
    #print("c_hat_bef_mid:", c_hat_tmp[:,19].mean())
    z = self.latent_predictor(h)
    # apply interventions (eval only)
    if getattr(self, '_interventions', None) and not self.training:
        c_hat = c_hat.clone()
        #mask_pop = logits[:, 18] > -0.23
        #mask_mid = logits[:, 19] < 1
        
        #mask_niche = logits[:, 20] < 0

        #mask = logits[:, 18] > logits[:, 19] # e.g. 0.0 → p(pop) > 0.5
        #c_hat[mask,19] += (3)

        for idx, scale in self._interventions.items():
            #print(f"idx {idx}")
            #print("before",idx, c_hat[:,idx].mean())
            #if idx==18:

                #logits[mask_pop, idx] += (scale)

            #elif idx==19:
                #logits[mask_mid,idx] += (scale)

            #elif idx==20:
            # logits_all: [num_users, num_concepts]
            #concept_std = logits[:,idx].std(dim=0)

            #logits[:,idx] += (scale)*concept_std
            c_hat[:,idx] *= (scale)

            #print("after:", idx, logits[:,idx].mean())
    #c_hat=torch.sigmoid(logits)
    #print(f"std_of_c_hat {c_hat.std(dim=0)}")
    #print("c_hat_after_pop:", c_hat[:,16].mean())
    #print("c_hat_after_mid:", c_hat[:,18].mean())

    #c_hat = torch.cat([
        #F.softmax(logits[:, :18],    dim=-1),   # beer styles
        #F.softmax(logits[:, 18:21], dim=-1),   # popularity tiers
        #F.softmax(logits[:, 21:24], dim=-1),   # era bins
        #], dim=-1)



        
    

    

        #c_hat = torch.cat([
        #F.softmax(c_hat[:, :16],    dim=-1),   # beer styles
        #F.softmax(c_hat[:, 16:19], dim=-1),   # popularity tiers
       # F.softmax(c_hat[:, 19:22], dim=-1),   # era bins
   # ], dim=-1)
    
    #c_hat = torch.cat([
        #F.softmax(logits[:, :16],    dim=-1),   # beer styles
        #F.softmax(logits[:, 16:19], dim=-1),   # popularity tiers
       # F.softmax(logits[:, 19:22], dim=-1),   # era bins
    #], dim=-1)
    # capture activations (eval only, when enabled)
   

    bottleneck = torch.cat([c_hat, z], dim=-1)
    #bottleneck = c_hat
    h_hat = self.reconstructor(bottleneck)
    return h, c_hat, z, h_hat
    #return h, c_hat, h_hat







# ═══════════════════════════════════════════════════════════════
# Config
# ═══════════════════════════════════════════════════════════════
config['metrics']      = ['Recall', 'NDCG', 'MRR', 'Hit',
                          'ItemCoverage', 'GiniIndex',
                          'AveragePopularity',
                          'TierExposure', 'ItemCoverageN',  "GenreExposure", "YearExposure"]
config['topk']         = [10]
config['valid_metric'] = 'NDCG@10'
config['eval_args'] = {
    'split': {'LS': 'valid_and_test'},
    'order': 'TO',
    'group_by': 'user',
    'mode': 'full'
}
config['tail_ratio']   = 0.2

L = build_lookups(dataset, config['dataset'])
config['popularity_pool'] = list(L['pop_set'])
config['mid_pool']        = list(L['mid_set'])
config['niche_pool']      = list(L['niche_set'])
config['item_to_genres']=dict(L['item_to_genres'])
config['genre_list']=list(L['genre_concepts'])

config['item_to_era']=dict(L['item_to_era'])

# ABV
#config['item_to_abv'] = dict(L['item_to_abv'])
 #Automatically get ABV category names
#config['abv_list'] = list(dict.fromkeys(L['item_to_abv'].values()))

# Style
#config['item_to_style'] = dict(L['item_to_style'])
#config['style_list'] = list(L['style_concepts'])


#c_hat_base, c_base = stop_capture(sasrec_model)


# ═══════════════════════════════════════════════════════════════
# Patch model (before any eval)
# ═══════════════════════════════════════════════════════════════
CBM_model.steer_concept_idx = None
CBM_model.steer_scale = 1.0
CBM_model.forward = types.MethodType(patched_forward, CBM_model)

concept_idx_pop = CBM_model.concept_names.index('modern')
concept_idx_mid = CBM_model.concept_names.index('mid')
concept_idx_niche = CBM_model.concept_names.index('niche')

#print(f"concept_idx_pop{concept_idx_pop}")
#print(f"concept_idx_mid{concept_idx_mid}")

# ═══════════════════════════════════════════════════════════════
# Baseline (no steering)
# ═══════════════════════════════════════════════════════════════
CBM_model._interventions = {}
#start_capture(cbm_sasrec)
result_baseline = evaluate_with_recbole(CBM_model, config, train_data, test_data)

# ═══════════════════════════════════════════════════════════════
# Steered
# ═══════════════════════════════════════════════════════════════
CBM_model._interventions = {
    concept_idx_pop: 0,
    concept_idx_mid: 1,
    concept_idx_niche: 1
}
#start_capture(cbm_sasrec)
result_steered = evaluate_with_recbole(CBM_model, config, train_data, test_data)
#c_hat_steer, c_steer = stop_capture(cbm_sasrec)

CBM_model._interventions = {}



# ═══════════════════════════════════════════════════════════════
# Results table
# ═══════════════════════════════════════════════════════════════
gap_before = result_baseline['pop_rate@10'] - result_baseline['mid_rate@10']
gap_after  = result_steered['pop_rate@10'] - result_steered['mid_rate@10']
print(f"\npop-mid gap: {gap_before:.4f} → {gap_after:.4f}")

print(f"\n{'Metric':<35} {'Baseline':>12} {'Steered':>12} {'Δ':>12}")
print("─" * 73)
for k in sorted(result_baseline.keys()):
    b, s = result_baseline[k], result_steered[k]
    print(f"{k:<35} {b:>12.4f} {s:>12.4f} {s - b:>+12.4f}")

## Zeroing-out top cocnepts

In [ ]:

@torch.no_grad()
def _encode_cbm(cbm_model, interaction):
    """Return predicted concepts c_hat and latent representation z."""
    h = cbm_model._encode(
        interaction[cbm_model.ITEM_SEQ],
        interaction[cbm_model.ITEM_SEQ_LEN],
    )
    c_hat = torch.sigmoid(cbm_model.concept_predictor(h))
    z = cbm_model.latent_predictor(h)
    return c_hat, z


@torch.no_grad()
def _recommend(cbm_model, interaction, c_hat, z, k=20):
    """Generate top-k recommendations from [c_hat ; z]."""
    bottleneck = torch.cat([c_hat, z], dim=-1)
    h_hat = cbm_model.reconstructor(bottleneck)
    scores = h_hat @ cbm_model.item_embedding.weight.T

    # Exclude padding and items already seen by the user.
    seen = set(
        interaction[cbm_model.ITEM_SEQ][0].detach().cpu().tolist()
    ) - {0}

    if seen:
        scores[0, list(seen)] = float("-inf")
    scores[0, 0] = float("-inf")

    return torch.topk(scores[0], k).indices.detach().cpu().tolist()


@torch.no_grad()
def steer_single_user(
    cbm_model,
    interaction,
    top_n=3,
    k=20,
    select_by="largest",   # "largest", "smallest", "random"
    seed=42,
    device=None,
    verbose=True
):
    """
    Zero out selected predicted concepts.

    select_by:
        "largest"  -> concepts with highest c_hat
        "smallest" -> concepts with lowest c_hat
        "random"   -> randomly selected concepts
    """

    if device is None:
        device = next(cbm_model.parameters()).device

    cbm_model.eval()
    interaction = interaction.to(device)

    # 1. Get original CBM representation
    c_hat, z = _encode_cbm(cbm_model, interaction)

    top_n = min(top_n, c_hat.shape[1])

    # ---------------------------------------------------------
    # 2. Choose which concepts to zero out
    # ---------------------------------------------------------
    if select_by == "largest":
        top_idx = torch.topk(
            c_hat[0],
            top_n,
            largest=True
        ).indices

    elif select_by == "smallest":
        top_idx = torch.topk(
            c_hat[0],
            top_n,
            largest=False
        ).indices

    elif select_by == "random":
        generator = torch.Generator()
        generator.manual_seed(seed)

        top_idx = torch.randperm(
            c_hat.shape[1],
            generator=generator
        )[:top_n].to(c_hat.device)

    else:
        raise ValueError(
            "select_by must be 'largest', 'smallest', or 'random'"
        )

    # 3. Original recommendations
    baseline = _recommend(
        cbm_model,
        interaction,
        c_hat,
        z,
        k=k
    )

    # 4. Zero out selected concepts
    c_steered = c_hat.clone()
    c_steered[0, top_idx] = 0.0

    # 5. Recommendations after intervention
    steered = _recommend(
        cbm_model,
        interaction,
        c_steered,
        z,
        k=k
    )

    # 6. Compare recommendation lists
    overlap = len(set(baseline) & set(steered))
    changed = k - overlap

    rbo_score = rbo.RankingSimilarity(
        baseline,
        steered
    ).rbo_ext(p=0.9)

    # Concept information
    concept_names = getattr(
        cbm_model,
        "concept_names",
        None
    )

    selected = []

    for idx in top_idx.tolist():

        name = (
            str(concept_names[idx])
            if concept_names is not None
            and idx < len(concept_names)
            else f"concept_{idx}"
        )

        selected.append(
            (
                idx,
                name,
                float(c_hat[0, idx])
            )
        )

    if verbose:

        print(f"\nSelection strategy: {select_by}")
        print("Selected concepts to zero out:")

        for idx, name, value in selected:
            print(
                f"  [{idx}] {name}: "
                f"{value:.4f}"
            )

        print(f"RBO: {rbo_score:.4f}")
        print(f"Overlap: {overlap}/{k}")
        print(f"Changed: {changed}/{k}")

    return {
        "select_by": select_by,
        "selected_concepts": selected,
        "baseline": baseline,
        "steered": steered,
        "rbo": rbo_score,
        "overlap": overlap,
        "changed": changed,
    }



@torch.no_grad()
def steer_population(
    cbm_model,
    data_loader,
    top_n=3,
    k=20,
    select_by="largest",
    seed=42,
    max_users=None,
    device=None
):
    """
    Apply concept zeroing to many users.

    select_by:
        "largest"
        "smallest"
        "random"
    """

    if device is None:
        device = next(cbm_model.parameters()).device

    rbo_scores = []
    changed_items = []
    n_users = 0

    for batch in data_loader:

        interaction = (
            batch[0]
            if isinstance(batch, (tuple, list))
            else batch
        )

        interaction = interaction.to(device)

        batch_size = interaction[
            cbm_model.ITEM_SEQ
        ].shape[0]

        for i in range(batch_size):

            result = steer_single_user(
                cbm_model,
                interaction[i:i + 1],
                top_n=top_n,
                k=k,
                select_by=select_by,
                seed=seed + n_users,
                device=device,
                verbose=False,
            )

            rbo_scores.append(
                result["rbo"]
            )

            changed_items.append(
                result["changed"]
            )

            n_users += 1

            if (
                max_users is not None
                and n_users >= max_users
            ):
                break

        if (
            max_users is not None
            and n_users >= max_users
        ):
            break

    rbo_scores = np.asarray(rbo_scores)
    changed_items = np.asarray(changed_items)

    print(f"\nSelection: {select_by}")
    print(f"Users: {n_users}")
    print(
        f"Mean RBO: "
        f"{rbo_scores.mean():.4f}"
    )
    print(
        f"Mean changed items: "
        f"{changed_items.mean():.2f}/{k}"
    )
    print(
        f"Users with >=1 changed item: "
        f"{(changed_items > 0).mean() * 100:.1f}%"
    )

    return {
        "select_by": select_by,
        "rbo": rbo_scores,
        "changed": changed_items,
        "n_users": n_users,
    }


In [ ]:
top_results = steer_population(
    CBM_model,
    test_data,
    top_n=3,
    k=20,
    select_by="largest" # largest, smallest, random
)

## using two concepts for steering

In [ ]:
def evaluate_with_recbole_sasrec(model, config, train_data, test_data):
    """Set steering on the model and run RecBole eval. Tier/genre/year exposure
    metrics are reported automatically alongside standard rec metrics."""
   

    

    trainer = Trainer(config, model)
    trainer.eval_collector = Collector(config)
    trainer.evaluator      = Evaluator(config)
    trainer.eval_collector.data_collect(train_data)

    result = trainer.evaluate(test_data, load_best_model=False, show_progress=False)

 
    return result
def evaluate_with_recbole(cbm, config, train_data, test_data,
                          concept_idx=None, scale=1.0):
    """Set steering on the model and run RecBole eval. Tier/genre/year exposure
    metrics are reported automatically alongside standard rec metrics."""
    cbm.steer_concept_idx = concept_idx
    #config['steer_concept_idx'] = concept_idx
    cbm.steer_scale       = scale
    #config['steer_scale']       = scale
   

    trainer = Trainer(config, cbm)
    trainer.eval_collector = Collector(config)
    trainer.evaluator      = Evaluator(config)
    trainer.eval_collector.data_collect(train_data)

    result = trainer.evaluate(test_data, load_best_model=False, show_progress=False)

    cbm.steer_concept_idx = None
    #config['steer_concept_idx'] = None
    cbm.steer_scale       = 1.0
    #config['steer_scale' ]      = 1.0
    return result